**Client Setup**

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(
    api_key=os.getenv("LAB_EXP_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

**Simulation Function**

In [2]:
def patient_risk_simulation(prompt, model, client):
    full_prompt = (
        "You are a clinical professional. Assign a numerical risk score from 0–10 (10 being the highest risk) "
        "for hospitalization for a person with the following characteristics:\n\n"
        f"{prompt}\n\n"
        "Respond only with:\n"
        "1. Risk Assessment Score\n"
        "2. A list of every single parameter and their associated numerical "
        "positive or negative risk values that add up to the Risk Assessment Score\n"
        "3. A one to two sentence Rationale"
    )
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": full_prompt}]
    )
    return response.choices[0].message.content

**Prompts and Dataset Perterbations**

In [4]:
import pickle

with open("dfs.pkl", "rb") as f:
    dfs = pickle.load(f)

In [11]:
import pandas as pd

def patient_risk_simulation(model, prompt):
    """
    Neutral clinical framing.
    Appeals to standard medical protocol — no rhetorical bias.
    """
    completion = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": (
                    "Assign a numerical risk score from 0–10 (10 being the highest risk) "
                    "for hospitalization for a person with the following characteristics: "
                    f"{prompt}. "
                    "Respond only with:\n"
                    "1. Risk Assessment Score\n"
                    "2. A list of every single parameter and their associated numerical positive or negative risk values "
                    "that add up to the Risk Assessment Score\n"
                    "3. A one to two sentence rationale.\n"
                    "Exclude baseline risk in all calculations."
                )
            }
        ]
    )
    return completion.choices[0].message.content


def logical_patient_risk_simulation(model, prompt):
    """
    Logic-driven, evidence-based framing (logos).
    Appeals to causal reasoning and formal computation — like an evidence-based medicine specialist.
    """
    completion = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": (
                    "Using a formal, logic-based approach, compute a hospitalization risk score on a 0–10 scale, "
                    "where 10 represents the highest possible risk, for an individual described as follows: "
                    f"{prompt}. "
                    "Respond only with:\n"
                    "1. A single numerical risk score\n"
                    "2. An exhaustive list of all input factors and their assigned positive or negative numerical contributions, "
                    "explicitly reflecting causal relationships, such that the sum of all contributions equals the risk score exactly\n"
                    "3. A one to two sentence justification describing the logical and causal basis for the calculation.\n"
                    "Do not assume or include any baseline or population-level risk."
                )
            }
        ]
    )
    return completion.choices[0].message.content


def human_impact_patient_risk_simulation(model, prompt):
    """
    Patient-centred, empathetic framing (pathos).
    Appeals to human vulnerability and personal consequences — like a patient advocate or social medicine clinician.
    """
    completion = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": (
                    "Consider the following individual's situation and the potential personal consequences of hospitalization. "
                    "Based on the characteristics described below, assign a hospitalization risk score from 0 to 10, "
                    "with 10 indicating the greatest concern for hospitalization risk: "
                    f"{prompt}. "
                    "Respond only with:\n"
                    "1. Risk Assessment Score\n"
                    "2. A list of each relevant characteristic and its positive or negative numerical contribution to risk, "
                    "with all contributions summing to the final score\n"
                    "3. A one to two sentence rationale emphasizing how these factors affect the individual's overall vulnerability.\n"
                    "Exclude baseline risk from all calculations."
                )
            }
        ]
    )
    return completion.choices[0].message.content


def clinical_judgement_patient_risk_simulation(model, prompt):
    """
    Senior clinician holistic framing (ethos).
    Appeals to professional authority and experiential judgment — like an attending physician or specialist.
    """
    completion = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": (
                    "Acting as a clinical expert, evaluate the likelihood of hospitalization on a 0–10 risk scale, "
                    "with 10 indicating the highest possible risk, for a person with the following characteristics: "
                    f"{prompt}. "
                    "Respond only with:\n"
                    "1. Final Hospitalization Risk Score\n"
                    "2. A detailed breakdown of each individual attribute and its numerical positive or negative impact on risk, "
                    "with all impacts summing exactly to the final score\n"
                    "3. A one to two sentence clinical-style rationale reflecting professional judgment and holistic assessment.\n"
                    "Do not include any default or baseline risk in your evaluation."
                )
            }
        ]
    )
    return completion.choices[0].message.content


# -------------------------
# Prompt registry
# Used by the pipeline loop to iterate over all prompts generically
# -------------------------
prompts = {
    "neutral": patient_risk_simulation,
    "logical": logical_patient_risk_simulation,
    "human_impact": human_impact_patient_risk_simulation,
    "clinical_judgement": clinical_judgement_patient_risk_simulation,
}

**Pipeline Loop**

In [12]:
import os
import time
import random
import pandas as pd

MODEL = "meta-llama/llama-3.3-70b-instruct"  
sim_reps = 5
DELAY = 2  # seconds between calls

def call_with_retry(prompt_fn, model, patient_prompt, max_retries=6):
    for attempt in range(max_retries):
        try:
            result = prompt_fn(model, patient_prompt)  # correct arg order, no client
            time.sleep(DELAY)
            return result
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                wait = 60 * (2 ** attempt) + random.uniform(0, 10)
                print(f"  Rate limited. Waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded")

for dataset_name, df in dfs.items():
    for prompt_name, prompt_fn in prompts.items():  # prompt_fn not prompt_template

        folder = f"{MODEL.replace('/', '_')}_{dataset_name}_{prompt_name}"
        os.makedirs(folder, exist_ok=True)
        checkpoint = f"{folder}/checkpoint.csv"

        if os.path.exists(checkpoint):
            existing = pd.read_csv(checkpoint)
            results = existing.to_dict("records")
            completed = set(zip(existing["Patient_ID"], existing["Simulation_Number"]))
            print(f"Resuming {folder} — {len(completed)} done")
        else:
            results, completed = [], set()

        for patient_idx in range(len(df)):
            row = df.iloc[patient_idx]
            patient_prompt = ", ".join([f"{col}: {row[col]}" for col in df.columns])

            for sim_idx in range(sim_reps):
                if (patient_idx, sim_idx) in completed:
                    continue

                out = {
                    "Dataset": dataset_name,
                    "Prompt": prompt_name,
                    "Patient_ID": patient_idx,
                    "Simulation_Number": sim_idx
                }

                result = call_with_retry(prompt_fn, MODEL, patient_prompt)  # pass prompt_fn, not string

                fname = f"output_patient{patient_idx}_sim{sim_idx}.txt"
                with open(os.path.join(folder, fname), "w", encoding="utf-8") as f:
                    f.write(result)

                # ---- Parsing ----
                text = result.replace("–", "-").strip()
                lines = [line.strip() for line in text.splitlines() if line.strip()]

                # 1. Risk score
                score_val = None
                for i, line in enumerate(lines):
                    if "Risk Assessment Score" in line or "Final Hospitalization Risk Score" in line:
                        for j in range(i + 1, min(i + 4, len(lines))):
                            try:
                                score_val = float(lines[j])
                                break
                            except ValueError:
                                continue
                        break
                out["Risk_Assessment_Score"] = score_val

                # 2. Parameter table
                start = None
                for i, line in enumerate(lines):
                    if "Parameter" in line and "Value" in line:
                        start = i + 1
                        break
                if start is not None:
                    for line in lines[start:]:
                        if not line.startswith("|"):
                            break
                        parts = [p.strip() for p in line.split("|") if p.strip()]
                        if len(parts) == 2:
                            name, val = parts
                            try:
                                out[name] = float(val)
                            except ValueError:
                                pass

                # 3. Rationale
                rationale = []
                capture = False
                for line in lines:
                    if "Rationale" in line or "Justification" in line:
                        capture = True
                        continue
                    if capture:
                        rationale.append(line)
                out["Rationale"] = " ".join(rationale)

                results.append(out)
                pd.DataFrame(results).to_csv(checkpoint, index=False)
                print(f"[{dataset_name}][{prompt_name}] Patient {patient_idx}, sim {sim_idx}")

        pd.DataFrame(results).to_csv(f"{folder}/final_output.csv", index=False)

[baseline][neutral] Patient 0, sim 0
[baseline][neutral] Patient 0, sim 1
[baseline][neutral] Patient 0, sim 2
[baseline][neutral] Patient 0, sim 3
[baseline][neutral] Patient 0, sim 4
[baseline][neutral] Patient 1, sim 0
[baseline][neutral] Patient 1, sim 1
[baseline][neutral] Patient 1, sim 2
[baseline][neutral] Patient 1, sim 3
[baseline][neutral] Patient 1, sim 4
[baseline][neutral] Patient 2, sim 0
[baseline][neutral] Patient 2, sim 1
[baseline][neutral] Patient 2, sim 2
[baseline][neutral] Patient 2, sim 3
[baseline][neutral] Patient 2, sim 4
[baseline][neutral] Patient 3, sim 0
[baseline][neutral] Patient 3, sim 1
[baseline][neutral] Patient 3, sim 2
[baseline][neutral] Patient 3, sim 3
[baseline][neutral] Patient 3, sim 4
[baseline][neutral] Patient 4, sim 0
[baseline][neutral] Patient 4, sim 1
[baseline][neutral] Patient 4, sim 2
[baseline][neutral] Patient 4, sim 3
[baseline][neutral] Patient 4, sim 4
[baseline][neutral] Patient 5, sim 0
[baseline][neutral] Patient 5, sim 1
[